# Step 1 — Training-Free Baseline
DINOv2 / DINOv3 / SAM · argmax · per-image & per-keypoint PCK · SPair-71k

In [11]:
# Cell 0 — Mount Drive + create folder structure
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/semantic_correspondence'
DIRS = [
    f'{DRIVE_ROOT}/datasets/SPair-71k',
    f'{DRIVE_ROOT}/datasets/PF-Pascal',
    f'{DRIVE_ROOT}/datasets/PF-Willow',
    f'{DRIVE_ROOT}/datasets/AP-10K',
    f'{DRIVE_ROOT}/weights',
    f'{DRIVE_ROOT}/weights/finetuned',
    f'{DRIVE_ROOT}/weights/finetuned/lora',
    f'{DRIVE_ROOT}/results/step1',
    f'{DRIVE_ROOT}/results/step2',
    f'{DRIVE_ROOT}/results/step3',
    f'{DRIVE_ROOT}/results/step4/lora',
    f'{DRIVE_ROOT}/results/step4/mnn',
    f'{DRIVE_ROOT}/results/step4/ensemble',
    f'{DRIVE_ROOT}/results/step4/ap10k',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print('Folders ready.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folders ready.


In [2]:
# Cell 1 — Install packages + clone repo for model architectures
!pip install -q timm scikit-learn pandas segment-anything

import os, subprocess
REPO_PATH = '/content/semantic_correspondence'
if not os.path.exists(REPO_PATH):
    # Replace URL with your actual repo URL
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'local-notebooks',
                    'https://github.com/Hesam-AHT/Semantic-Correspondence-with-Visual-Foundation-Models.git',
                    REPO_PATH], check=False)
    print('Repo cloned.')
else:
    print('Repo already present.')

import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('sys.path updated.')

Repo cloned.
sys.path updated.


In [ ]:
# Cell 2 — Download SPair-71k (guarded)
SPAIR_DIR = f'{DRIVE_ROOT}/datasets/SPair-71k'
SPAIR_TAR = f'{SPAIR_DIR}/SPair-71k.tar.gz'
SPAIR_CHECK = f'{SPAIR_DIR}/JPEGImages'

if not os.path.exists(SPAIR_CHECK):
    print('Downloading SPair-71k ...')
    !wget -q -O "{SPAIR_TAR}" http://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz
    !tar -xzf "{SPAIR_TAR}" -C "{SPAIR_DIR}"
    print('SPair-71k extracted.')
else:
    print('SPair-71k already present.')

SPair-71k extracted.


In [4]:
# Cell 3 — Download pretrained weights (guarded)
import subprocess

DINOV2_W = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
if not os.path.exists(DINOV2_W):
    print('Downloading DINOv2 weights...')
    subprocess.run(['wget', '-q', '-O', DINOV2_W,
        'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth'])
    print('DINOv2 weights saved.')
else:
    print('DINOv2 weights already present.')

SAM_W = f'{DRIVE_ROOT}/weights/sam_vit_b.pth'
if not os.path.exists(SAM_W):
    print('Downloading SAM ViT-B weights...')
    subprocess.run(['wget', '-q', '-O', SAM_W,
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'])
    print('SAM weights saved.')
else:
    print('SAM weights already present.')

DINOV3_W = f'{DRIVE_ROOT}/weights/dinov3_vitb16_pretrain.pth'
if not os.path.exists(DINOV3_W):
    print('Downloading DINOv3 weights...')
    subprocess.run(['wget', '-q', '-O', DINOV3_W,
        'https://dinov3.llamameta.net/dinov3_vitb16/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiaXNuZXB2NWJyMmpvcXM4N3Y2dTlrNDFlIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3Nzk0ODQ1NTF9fX1dfQ__&Signature=s4mL8ILUFqU0eE1KBha8ryjjPJzobeZNek7-G~skvFiuTaDOvBw47eWenOENnHNWN8iPISuCsj5zQJt7pZRGMDEbbq1D3yTQOuNjDFRvX95Xx6wtNYb7TxkP3xQJD-OEWQu3L-5x-wHVW-m-nLTmKyMN1v~QTFoWm6OWixvAz3qVpyK4Ho2xeShlUpCdX19vONCk2EPGwj0akN4AgI5sAaLz9VmY4VRXygrEARhMhhGqBs0FuaKZXdkggMOW~gH3NuEia3BJn84jQLKPTbu1HZa8flz0NQ5~ZvetXOFS-SbrpaIEuwZiLTVdP1djkV8tZsNugBue-savfiLrnJpgvw__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=914567197795521'])
    print('DINOv3 weights saved.')
else:
    print('DINOv3 weights already present.')

DINOv2 weights already present.
SAM weights already present.
DINOv3 weights already present.


In [12]:
# Cell 4 — Imports + config
import torch, numpy as np, json, time, os, math
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from collections import defaultdict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

SPAIR_BASE      = f'{DRIVE_ROOT}/datasets/SPair-71k/SPair-71k'
PAIR_ANN_PATH   = f'{SPAIR_BASE}/PairAnnotation'
LAYOUT_PATH     = f'{SPAIR_BASE}/Layout'
IMAGE_PATH      = f'{SPAIR_BASE}/JPEGImages'
DATASET_SIZE    = 'large'
PCK_ALPHA       = 0.1
THRESHOLDS      = [0.05, 0.1, 0.2]
RESULTS_DIR     = f'{DRIVE_ROOT}/results/step1'
IMG_SIZE_DINOV2 = 518
IMG_SIZE_DINOV3 = 512
IMG_SIZE_SAM    = 512

Device: cuda


In [13]:
# Cell 5 — Inline utility functions
# ----- Data utilities -----
class Normalize:
    def __init__(self, image_keys):
        self.image_keys = image_keys
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, sample):
        for key in self.image_keys:
            sample[key] /= 255.0
            sample[key] = self.normalize(sample[key])
        return sample

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

import glob
class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.datatype = datatype
        self.pck_alpha = pck_alpha
        self.ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt')).read().split('\n')
        self.ann_files = self.ann_files[:len(self.ann_files) - 1]
        self.pair_ann_path = pair_ann_path
        self.image_path = image_path
        self.transform = Normalize(['src_img', 'trg_img'])
    def __len__(self): return len(self.ann_files)
    def __getitem__(self, idx):
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            ann = json.load(f)
        category = ann['category']
        src_img = read_img(os.path.join(self.image_path, category, ann['src_imname']))
        trg_img = read_img(os.path.join(self.image_path, category, ann['trg_imname']))
        trg_bbox = ann['trg_bndbox']
        sample = {'src_imname': ann['src_imname'], 'trg_imname': ann['trg_imname'],
                  'src_imsize': src_img.size(), 'trg_imsize': trg_img.size(),
                  'trg_bbox': trg_bbox, 'category': category,
                  'src_img': src_img, 'trg_img': trg_img,
                  'src_kps': torch.tensor(ann['src_kps']).float(),
                  'trg_kps': torch.tensor(ann['trg_kps']).float(),
                  'kps_ids': ann['kps_ids']}
        return self.transform(sample)

# ----- Feature extraction -----
def extract_dense_features(model, img_tensor):
    with torch.no_grad():
        fd = model.forward_features(img_tensor)
        pt = fd['x_norm_patchtokens']
        B, N, D = pt.shape
        H = W = int(N ** 0.5)
        return pt.reshape(B, H, W, D)

def extract_dense_features_SAM(model, img_tensor, image_size=512):
    import torch.nn as nn
    from torch.cuda.amp import autocast
    with torch.no_grad():
        img_r = F.interpolate(img_tensor, size=(image_size, image_size), mode='bilinear', align_corners=False)
        if image_size != 1024:
            orig_pe = model.image_encoder.pos_embed
            ng = image_size // 16
            pe_r = F.interpolate(orig_pe.permute(0,3,1,2), size=(ng,ng),
                                 mode='bicubic', align_corners=False).permute(0,2,3,1)
            model.image_encoder.pos_embed = nn.Parameter(pe_r, requires_grad=False)
        with autocast():
            emb = model.image_encoder(img_r)
        if image_size != 1024:
            model.image_encoder.pos_embed = orig_pe
        return emb.permute(0, 2, 3, 1)

# ----- Coordinate utilities -----
def pixel_to_patch_coord(x, y, original_size, patch_size=14, resized_size=518):
    sx = resized_size / original_size[0]
    sy = resized_size / original_size[1]
    px = int(x * sx // patch_size)
    py = int(y * sy // patch_size)
    mx = resized_size // patch_size - 1
    return min(max(px, 0), mx), min(max(py, 0), mx)

def patch_to_pixel_coord(patch_x, patch_y, original_size, patch_size=14, resized_size=518):
    xr = patch_x * patch_size + patch_size / 2
    yr = patch_y * patch_size + patch_size / 2
    return xr * original_size[0] / resized_size, yr * original_size[1] / resized_size

# ----- Matching -----
def find_best_match_argmax(s, width):
    idx = s.argmax().item()
    return idx % width, idx // width

# ----- PCK -----
def compute_pck_spair71k(pred_points, gt_points, bbox, threshold):
    pred = np.array(pred_points)
    gt   = np.array(gt_points)
    dist = np.sqrt(np.sum((pred - gt) ** 2, axis=1))
    norm = max(bbox[2] - bbox[0], bbox[3] - bbox[1])
    nd   = dist / norm
    return float(np.mean(nd <= threshold) * 100), (nd <= threshold), nd

# ----- Evaluation -----
def run_evaluate(model, dataset, device, thresholds, patch_size, resized_size,
                 feat_fn=None, early_stop=False):
    if feat_fn is None:
        feat_fn = lambda m, t: extract_dense_features(m, t)
    per_img, all_kp = [], []
    t0 = time.time()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                  size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                  size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = feat_fn(model, src_t)
            tf = feat_fn(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps  = sample['src_kps'].numpy()
            trg_kps  = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            kps_ids  = sample['kps_ids']
            norm_dist = max(trg_bbox[2] - trg_bbox[0], trg_bbox[3] - trg_bbox[1])
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, resized_size)
                sf_q = sf[0, py, px, :]
                sim = F.cosine_similarity(sf_q.unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, resized_size)
                preds.append([rx, ry])
                gt_x, gt_y = float(trg_kps[i, 0]), float(trg_kps[i, 1])
                nd = float(np.sqrt((rx - gt_x)**2 + (ry - gt_y)**2) / norm_dist)
                all_kp.append({
                    'category':   sample['category'],
                    'src_imname': str(sample['src_imname']),
                    'trg_imname': str(sample['trg_imname']),
                    'kp_id':      kps_ids[i] if i < len(kps_ids) else i,
                    'pred':       [float(rx), float(ry)],
                    'gt':         [gt_x, gt_y],
                    'norm_dist':  nd,
                    'correct':    {str(thr): bool(nd <= thr) for thr in thresholds},
                })
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks,
                            'src_imname': str(sample['src_imname']),
                            'trg_imname': str(sample['trg_imname'])})
            if (idx+1) % 200 == 0: print(f'  {idx+1} pairs...')
            if early_stop and idx >= 50: break
    return per_img, all_kp, time.time() - t0

def save_exp_results(per_img, all_kp, name, thresholds, results_dir):
    print('  -- Per-image PCK --')
    stats = {}
    for thr in thresholds:
        pcks = [m['pck_scores'][thr] for m in per_img]
        stats[f'pck@{thr:.2f}'] = {'mean': float(np.mean(pcks)), 'std': float(np.std(pcks))}
        print(f'    PCK@{thr:.2f}: {np.mean(pcks):.2f}% +/- {np.std(pcks):.2f}%')
    with open(os.path.join(results_dir, f'{name}_per_image.json'), 'w') as f:
        json.dump({'name': name, 'n_pairs': len(per_img), 'stats': stats, 'per_image': per_img}, f, indent=2)
    print('  -- Per-keypoint PCK --')
    for thr in thresholds:
        correct = [kp['correct'][str(thr)] for kp in all_kp]
        print(f'    PCK@{thr:.2f}: {float(np.mean(correct)*100):.2f}%  ({sum(correct)}/{len(correct)} kps)')
    with open(os.path.join(results_dir, f'{name}_per_keypoint.json'), 'w') as f:
        json.dump({'name': name, 'n_keypoints': len(all_kp), 'per_keypoint': all_kp}, f, indent=2)
    print(f'  Saved: {name}_per_image.json + {name}_per_keypoint.json')
    return stats

print('Utility functions loaded.')

Utility functions loaded.


In [6]:
# Cell 6 — Load DINOv2 model
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as dinov2_vit_base

dinov2 = dinov2_vit_base(
    img_size=(518, 518), patch_size=14,
    num_register_tokens=0, block_chunks=0, init_values=1.0
).to(device)
ckpt = torch.load(DINOV2_W, map_location=device)
dinov2.load_state_dict(ckpt, strict=True)
dinov2.eval()
print('DINOv2 loaded.')

test_ds = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='test')
print(f'Test set: {len(test_ds)} pairs')

/content/semantic_correspondence/src/models/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/semantic_correspondence/src/models/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/semantic_correspondence/src/models/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


DINOv2 loaded.
Test set: 12234 pairs


In [ ]:
# Cell 7 — Exp 1.1: DINOv2 last layer + argmax
print('=== Exp 1.1: DINOv2 last layer + argmax ===')
res_11, kp_11, t = run_evaluate(dinov2, test_ds, device, THRESHOLDS,
                                 patch_size=14, resized_size=IMG_SIZE_DINOV2)
stats_11 = save_exp_results(res_11, kp_11, 'exp1_1_dinov2_argmax', THRESHOLDS, RESULTS_DIR)
print(f'Time: {t:.1f}s')

=== Exp 1.1: DINOv2 last layer + argmax ===
  200 pairs...
  400 pairs...
  600 pairs...
  800 pairs...
  1000 pairs...
  1200 pairs...
  1400 pairs...
  1600 pairs...
  1800 pairs...
  2000 pairs...
  2200 pairs...
  2400 pairs...
  2600 pairs...
  2800 pairs...
  3000 pairs...
  3200 pairs...
  3400 pairs...
  3600 pairs...
  3800 pairs...
  4000 pairs...
  4200 pairs...
  4400 pairs...
  4600 pairs...
  4800 pairs...
  5000 pairs...
  5200 pairs...
  5400 pairs...
  5600 pairs...
  5800 pairs...
  6000 pairs...
  6200 pairs...
  6400 pairs...
  6600 pairs...
  6800 pairs...
  7000 pairs...
  7200 pairs...
  7400 pairs...
  7600 pairs...
  7800 pairs...
  8000 pairs...
  8200 pairs...
  8400 pairs...
  8600 pairs...
  8800 pairs...
  9000 pairs...
  9200 pairs...
  9400 pairs...
  9600 pairs...
  9800 pairs...
  10000 pairs...
  10200 pairs...
  10400 pairs...
  10600 pairs...
  10800 pairs...
  11000 pairs...
  11200 pairs...
  11400 pairs...
  11600 pairs...
  11800 pairs...
  1200

In [7]:
# Cell 10 — Load DINOv3
try:
    from src.models.dinov3.dinov3.models.vision_transformer import vit_base as dinov3_vit_base
    dinov3 = dinov3_vit_base(
        img_size=(512, 512), patch_size=16,
        n_storage_tokens=4, mask_k_bias=True,
        layerscale_init=1.0e-05, norm_layer='layernormbf16',
    ).to(device)
    ckpt3 = torch.load(DINOV3_W, map_location=device)
    dinov3.load_state_dict(ckpt3, strict=True)
    dinov3.eval()
    DINOV3_OK = True
    print('DINOv3 loaded.')
except Exception as e:
    print(f'DINOv3 not available ({e}). Skipping DINOv3 experiments.')
    DINOV3_OK = False

DINOv3 loaded.


In [8]:
# Cell 11 — Exp 1.2: DINOv3 last layer + argmax
if DINOV3_OK:
    print('=== Exp 1.2: DINOv3 last layer + argmax ===')
    res_12, kp_12, t = run_evaluate(dinov3, test_ds, device, THRESHOLDS,
                                     patch_size=16, resized_size=IMG_SIZE_DINOV3)
    stats_12 = save_exp_results(res_12, kp_12, 'exp1_2_dinov3_argmax', THRESHOLDS, RESULTS_DIR)
    print(f'Time: {t:.1f}s')
else:
    stats_12 = None
    print('Skipped.')

=== Exp 1.2: DINOv3 last layer + argmax ===
  200 pairs...
  400 pairs...
  600 pairs...
  800 pairs...
  1000 pairs...
  1200 pairs...
  1400 pairs...
  1600 pairs...
  1800 pairs...
  2000 pairs...
  2200 pairs...
  2400 pairs...
  2600 pairs...
  2800 pairs...
  3000 pairs...
  3200 pairs...
  3400 pairs...
  3600 pairs...
  3800 pairs...
  4000 pairs...
  4200 pairs...
  4400 pairs...
  4600 pairs...
  4800 pairs...
  5000 pairs...
  5200 pairs...
  5400 pairs...
  5600 pairs...
  5800 pairs...
  6000 pairs...
  6200 pairs...
  6400 pairs...
  6600 pairs...
  6800 pairs...
  7000 pairs...
  7200 pairs...
  7400 pairs...
  7600 pairs...
  7800 pairs...
  8000 pairs...
  8200 pairs...
  8400 pairs...
  8600 pairs...
  8800 pairs...
  9000 pairs...
  9200 pairs...
  9400 pairs...
  9600 pairs...
  9800 pairs...
  10000 pairs...
  10200 pairs...
  10400 pairs...
  10600 pairs...
  10800 pairs...
  11000 pairs...
  11200 pairs...
  11400 pairs...
  11600 pairs...
  11800 pairs...
  1200

In [ ]:
# Cell 12 — Load SAM
from src.models.segment_anything.segment_anything import sam_model_registry
sam = sam_model_registry['vit_b'](checkpoint=SAM_W).to(device)
sam.eval()
print('SAM loaded.')

SAM loaded.


In [ ]:
# Cell 13 — Exp 1.3: SAM last layer + argmax
print('=== Exp 1.3: SAM last layer + argmax ===')
sam_feat_fn = lambda m, t: extract_dense_features_SAM(m, t, image_size=IMG_SIZE_SAM)
res_13, kp_13, t = run_evaluate(sam, test_ds, device, THRESHOLDS,
                                 patch_size=16, resized_size=IMG_SIZE_SAM,
                                 feat_fn=sam_feat_fn)
stats_13 = save_exp_results(res_13, kp_13, 'exp1_3_sam_argmax', THRESHOLDS, RESULTS_DIR)
print(f'Time: {t:.1f}s')

=== Exp 1.3: SAM last layer + argmax ===


/tmp/ipykernel_468/544394671.py:65: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  200 pairs...
  400 pairs...
  600 pairs...
  800 pairs...
  1000 pairs...
  1200 pairs...
  1400 pairs...
  1600 pairs...
  1800 pairs...
  2000 pairs...
  2200 pairs...
  2400 pairs...
  2600 pairs...
  2800 pairs...
  3000 pairs...
  3200 pairs...
  3400 pairs...
  3600 pairs...
  3800 pairs...
  4000 pairs...
  4200 pairs...
  4400 pairs...
  4600 pairs...
  4800 pairs...
  5000 pairs...
  5200 pairs...
  5400 pairs...
  5600 pairs...
  5800 pairs...
  6000 pairs...
  6200 pairs...
  6400 pairs...
  6600 pairs...
  6800 pairs...
  7000 pairs...
  7200 pairs...
  7400 pairs...
  7600 pairs...
  7800 pairs...
  8000 pairs...
  8200 pairs...
  8400 pairs...
  8600 pairs...
  8800 pairs...
  9000 pairs...
  9200 pairs...
  9400 pairs...
  9600 pairs...
  9800 pairs...
  10000 pairs...
  10200 pairs...
  10400 pairs...
  10600 pairs...
  10800 pairs...
  11000 pairs...
  11200 pairs...
  11400 pairs...
  11600 pairs...
  11800 pairs...
  12000 pairs...
  12200 pairs...
  -- Per-image P

In [22]:
# Cell 14 -- Summary table
def load_stats(var, fname):
    if var is not None:
        return var
    p = os.path.join(RESULTS_DIR, fname)
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)['stats']
    return None

rows = [
    {'Exp': '1.1', 'Backbone': 'DINOv2', 'Stats': load_stats(stats_11 if 'stats_11' in dir() else None, 'exp1_1_dinov2_argmax_per_image.json')},
    {'Exp': '1.2', 'Backbone': 'DINOv3', 'Stats': load_stats(stats_12 if 'stats_12' in dir() else None, 'exp1_2_dinov3_argmax_per_image.json')},
    {'Exp': '1.3', 'Backbone': 'SAM',    'Stats': load_stats(stats_13 if 'stats_13' in dir() else None, 'exp1_3_sam_argmax_per_image.json')},
]
table = []
for r in rows:
    if r['Stats'] is None:
        table.append({'Exp': r['Exp'], 'Backbone': r['Backbone'],
                      'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})
    else:
        s = r['Stats']
        table.append({'Exp': r['Exp'], 'Backbone': r['Backbone'],
                      'PCK@0.05': f"{s.get('pck@0.05', {}).get('mean', 0):.2f}%",
                      'PCK@0.10': f"{s.get('pck@0.10', {}).get('mean', 0):.2f}%",
                      'PCK@0.20': f"{s.get('pck@0.20', {}).get('mean', 0):.2f}%"})
df = pd.DataFrame(table)
print('=== Step 1 Results ===')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/step1_summary.csv', index=False)
print(f'Summary saved to {RESULTS_DIR}/step1_summary.csv')

=== Step 1 Results ===
Exp Backbone PCK@0.05 PCK@0.10 PCK@0.20
1.1   DINOv2   35.61%   51.90%   67.64%
1.2   DINOv3   32.38%   48.31%   62.86%
1.3      SAM   11.72%   20.45%   33.39%
Summary saved to /content/drive/MyDrive/semantic_correspondence/results/step1/step1_summary.csv
